In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

In [2]:
data = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\ml_dataset.csv")
data.head().T

,0,1,2,3,4
year,2012,2012,2012,2012,2012
state_code,NY,IN,SC,IN,AL
provider_type,General Short-Term (includes CAH),General Short-Term (includes CAH),Psychiatric Hospital,Psychiatric Hospital,General Short-Term (includes CAH)
ccn_facility_type,Short-Term Hospital,Short-Term Hospital,Psychiatric Hospital,Psychiatric Hospital,Short-Term Hospital
number_of_beds,66.0,66.0,66.0,66.0,25.0
total_bed_days_available,23424.0,23424.0,23424.0,23424.0,2196.0
occupancy_rate,0.519244,0.519244,0.519244,0.519244,0.147692
total_discharges__v___xviii___xix___unknown_,1659.0,1659.0,1659.0,1659.0,62.0
total_days__v___xviii___xix___unknown_,11979.0,11979.0,11979.0,11979.0,288.0
fte___employees_on_payroll,295.42,295.42,295.42,295.42,52.5


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69555 entries, 0 to 69554
Data columns (total 51 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   year                                          69555 non-null  int64  
 1   state_code                                    69555 non-null  object 
 2   provider_type                                 69555 non-null  object 
 3   ccn_facility_type                             69555 non-null  object 
 4   number_of_beds                                69555 non-null  float64
 5   total_bed_days_available                      69555 non-null  float64
 6   occupancy_rate                                69555 non-null  float64
 7   total_discharges__v___xviii___xix___unknown_  69555 non-null  float64
 8   total_days__v___xviii___xix___unknown_        69555 non-null  float64
 9   fte___employees_on_payroll                    69555 non-null 

In [4]:
# Sort by year
data = data.sort_values(by='year')

train_data = data[data['year'] <= 2019]
test_data = data[data['year'] > 2019]

In [6]:
targets = ['profit_margin_calc', 'charity_care_ratio']

In [7]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "ExtraTrees": ExtraTreesRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR()
}

In [8]:
def build_pipeline(X):
    cat_cols = X.select_dtypes(include=['object']).columns
    num_cols = X.select_dtypes(include=[np.number]).columns

    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ])

    return preprocessor

In [11]:
results = []

for target in targets:
    print("\nTARGET:", target, "\n")

    # Drop target from features to avoid leakage
    X_train = train_data.drop(targets, axis=1)
    y_train = train_data[target]

    X_test = test_data.drop(targets, axis=1)
    y_test = test_data[target]

    preprocessor = build_pipeline(X_train)

    for name, model in models.items():
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        # Train
        pipe.fit(X_train, y_train)

        # Predictions
        y_train_pred = pipe.predict(X_train)
        y_test_pred = pipe.predict(X_test)

        # Metrics
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        train_mse = mean_squared_error(y_train, y_train_pred)
        test_mse = mean_squared_error(y_test, y_test_pred)

        # Store results
        results.append({
            'Target': target,
            'Model': name,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Train_MSE': train_mse,
            'Test_MSE': test_mse
        })

        print(name)
        print("Train R2:", round(train_r2, 4), "MSE:", round(train_mse, 4))
        print("Test  R2:", round(test_r2, 4), "MSE:", round(test_mse, 4))
        print("-" * 50)

# Create dataframe
results_df = pd.DataFrame(results)

# Sort results
results_df = results_df.sort_values(
    by=['Target', 'Test_R2'],
    ascending=[True, False]
).reset_index(drop=True)

# Round values
results_df = results_df.round({
    'Train_R2': 4,
    'Test_R2': 4,
    'Train_MSE': 2,
    'Test_MSE': 2
})

# Overfitting gap
results_df['Overfit_Gap'] = (results_df['Train_R2'] - results_df['Test_R2']).round(4)

# Best models per target
best_models = results_df.loc[results_df.groupby('Target')['Test_R2'].idxmax()]

results_df
best_models


TARGET: profit_margin_calc 

Linear
Train R2: 0.3487 MSE: 0.02
Test  R2: 0.2922 MSE: 0.0288
--------------------------------------------------


c:\Users\manju\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.44776e-23): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Ridge
Train R2: 0.3487 MSE: 0.02
Test  R2: 0.2922 MSE: 0.0288
--------------------------------------------------
Lasso
Train R2: 0.2194 MSE: 0.024
Test  R2: 0.1996 MSE: 0.0326
--------------------------------------------------
ElasticNet
Train R2: 0.2194 MSE: 0.024
Test  R2: 0.1984 MSE: 0.0326
--------------------------------------------------
DecisionTree
Train R2: 1.0 MSE: 0.0
Test  R2: 0.8124 MSE: 0.0076
--------------------------------------------------
RandomForest
Train R2: 0.9925 MSE: 0.0002
Test  R2: 0.9301 MSE: 0.0028
--------------------------------------------------
ExtraTrees
Train R2: 1.0 MSE: 0.0
Test  R2: 0.9155 MSE: 0.0034
--------------------------------------------------
GradientBoosting
Train R2: 0.8634 MSE: 0.0042
Test  R2: 0.8377 MSE: 0.0066
--------------------------------------------------
KNN
Train R2: 0.8315 MSE: 0.0052
Test  R2: 0.6543 MSE: 0.0141
--------------------------------------------------
SVR
Train R2: 0.432 MSE: 0.0175
Test  R2: 0.4568 MSE: 0.0221
--

c:\Users\manju\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.44776e-23): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Ridge
Train R2: 0.622 MSE: 0.0003
Test  R2: 0.571 MSE: 0.0003
--------------------------------------------------
Lasso
Train R2: 0.448 MSE: 0.0005
Test  R2: 0.321 MSE: 0.0005
--------------------------------------------------
ElasticNet
Train R2: 0.4487 MSE: 0.0005
Test  R2: 0.3234 MSE: 0.0005
--------------------------------------------------
DecisionTree
Train R2: 1.0 MSE: 0.0
Test  R2: 0.9837 MSE: 0.0
--------------------------------------------------
RandomForest
Train R2: 0.9994 MSE: 0.0
Test  R2: 0.9898 MSE: 0.0
--------------------------------------------------
ExtraTrees
Train R2: 1.0 MSE: 0.0
Test  R2: 0.9881 MSE: 0.0
--------------------------------------------------
GradientBoosting
Train R2: 0.9954 MSE: 0.0
Test  R2: 0.9892 MSE: 0.0
--------------------------------------------------
KNN
Train R2: 0.6309 MSE: 0.0003
Test  R2: 0.1607 MSE: 0.0007
--------------------------------------------------
SVR
Train R2: -5.0658 MSE: 0.0053
Test  R2: -5.8319 MSE: 0.0053
-----------------

,Target,Model,Train_R2,Test_R2,Train_MSE,Test_MSE,Overfit_Gap
0,charity_care_ratio,RandomForest,0.9994,0.9898,0.0,0.0,0.0096
10,profit_margin_calc,RandomForest,0.9925,0.9301,0.0,0.0,0.0624


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# sort and split
data = data.sort_values(by='year')

train_data = data[data['year'] <= 2019]
test_data = data[data['year'] > 2019]

targets = ['profit_margin_calc', 'charity_care_ratio']

# preprocessing
def build_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object']).columns
    num_cols = X.select_dtypes(include=[np.number]).columns

    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ])

# models and grids
models_and_params = {
    "Linear": (
        LinearRegression(),
        {}
    ),
    "Ridge": (
        Ridge(),
        {'model__alpha': [0.1, 1.0, 10]}
    ),
    "Lasso": (
        Lasso(),
        {'model__alpha': [0.001, 0.01, 0.1]}
    ),
    "ElasticNet": (
        ElasticNet(),
        {
            'model__alpha': [0.001, 0.01],
            'model__l1_ratio': [0.3, 0.5, 0.7]
        }
    ),
    "DecisionTree": (
        DecisionTreeRegressor(random_state=42),
        {
            'model__max_depth': [5, 8, 10],
            'model__min_samples_leaf': [1, 5, 10]
        }
    ),
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            'model__n_estimators': [100, 200],
            'model__max_depth': [8, 10],
            'model__min_samples_leaf': [1, 5]
        }
    ),
    "ExtraTrees": (
        ExtraTreesRegressor(random_state=42),
        {
            'model__n_estimators': [100, 200],
            'model__max_depth': [8, 10],
            'model__min_samples_leaf': [1, 5]
        }
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(),
        {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [3, 5]
        }
    ),
    "KNN": (
        KNeighborsRegressor(),
        {
            'model__n_neighbors': [5, 10],
            'model__weights': ['uniform', 'distance']
        }
    ),
    "SVR": (
        SVR(),
        {
            'model__C': [1, 10],
            'model__kernel': ['rbf'],
            'model__gamma': ['scale']
        }
    )
}

# cross validation
tscv = TimeSeriesSplit(n_splits=5)

results = []

# training loop
for target in targets:
    print("\nTARGET:", target, "\n")

    X_train = train_data.drop(targets, axis=1)
    y_train = train_data[target]

    X_test = test_data.drop(targets, axis=1)
    y_test = test_data[target]

    preprocessor = build_preprocessor(X_train)

    for name, (model, param_grid) in models_and_params.items():

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        grid = GridSearchCV(
            pipe,
            param_grid,
            cv=tscv,
            scoring='r2',
            n_jobs=-1
        )

        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_

        y_train_pred = best_model.predict(X_train)
        y_test_pred = best_model.predict(X_test)

        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        train_mse = mean_squared_error(y_train, y_train_pred)
        test_mse = mean_squared_error(y_test, y_test_pred)

        train_mae = mean_absolute_error(y_train, y_train_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)

        results.append({
            'Target': target,
            'Model': name,
            'Best_Params': grid.best_params_,
            'CV_Score': grid.best_score_,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Train_MSE': train_mse,
            'Test_MSE': test_mse,
            'Train_MAE': train_mae,
            'Test_MAE': test_mae
        })

        print(name)
        print("Best Params:", grid.best_params_)
        print("CV R2:", round(grid.best_score_, 4))
        print("Train R2:", round(train_r2, 4), "Test R2:", round(test_r2, 4))
        print("-" * 50)

# results dataframe
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=['Target', 'Test_R2'],
    ascending=[True, False]
).reset_index(drop=True)

results_df = results_df.round(4)

results_df['Overfit_Gap'] = (results_df['Train_R2'] - results_df['Test_R2']).round(4)

print("\nFINAL RESULTS\n")
print(results_df)

# best models
best_models = results_df.loc[results_df.groupby('Target')['Test_R2'].idxmax()]

print("\nBEST MODELS\n")
print(best_models)



TARGET: profit_margin_calc 

Linear
Best Params: {}
CV R2: 0.3281
Train R2: 0.3487 Test R2: 0.2922
--------------------------------------------------


c:\Users\manju\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.39087e-22): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Ridge
Best Params: {'model__alpha': 10}
CV R2: 0.329
Train R2: 0.3487 Test R2: 0.2922
--------------------------------------------------


c:\Users\manju\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.969e+01, tolerance: 1.565e-01
  model = cd_fast.enet_coordinate_descent(


Lasso
Best Params: {'model__alpha': 0.001}
CV R2: 0.312
Train R2: 0.3281 Test R2: 0.2829
--------------------------------------------------


c:\Users\manju\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.650e-01, tolerance: 1.565e-01
  model = cd_fast.enet_coordinate_descent(


ElasticNet
Best Params: {'model__alpha': 0.001, 'model__l1_ratio': 0.3}
CV R2: 0.323
Train R2: 0.3407 Test R2: 0.2902
--------------------------------------------------
DecisionTree
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 10}
CV R2: 0.8341
Train R2: 0.9285 Test R2: 0.8716
--------------------------------------------------
RandomForest
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
CV R2: 0.8862
Train R2: 0.966 Test R2: 0.9144
--------------------------------------------------
ExtraTrees
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
CV R2: 0.7765
Train R2: 0.8474 Test R2: 0.777
--------------------------------------------------
GradientBoosting
Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200}
CV R2: 0.9088
Train R2: 0.9901 Test R2: 0.9437
--------------------------------------------------
KNN
Best Params: {'model__n_neig

c:\Users\manju\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.39087e-22): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Ridge
Best Params: {'model__alpha': 10}
CV R2: 0.6072
Train R2: 0.6219 Test R2: 0.5723
--------------------------------------------------
Lasso
Best Params: {'model__alpha': 0.001}
CV R2: 0.5241
Train R2: 0.5393 Test R2: 0.4721
--------------------------------------------------
ElasticNet
Best Params: {'model__alpha': 0.001, 'model__l1_ratio': 0.3}
CV R2: 0.5476
Train R2: 0.5643 Test R2: 0.5085
--------------------------------------------------
DecisionTree
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 5}
CV R2: 0.9872
Train R2: 0.998 Test R2: 0.9851
--------------------------------------------------
RandomForest
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
CV R2: 0.9928
Train R2: 0.9991 Test R2: 0.9892
--------------------------------------------------
ExtraTrees
Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
CV R2: 0.9662
Train R2: 0.9767 Test R2: 0.9605
------------

In [ ]:
# Feature importance for best model of profit_margin_calc
best_profit_model_name = best_models[best_models['Target'] == 'profit_margin_calc']['Model'].values[0]
best_profit_model_params = best_models[best_models['Target'] == 'profit_margin_calc']['Best_Params'].values[0]
print("Best Model for profit_margin_calc:", best_profit_model_name)
print("Best Hyperparameters:", best_profit_model_params)

NameError: name 'best_models' is not defined

In [13]:
# FEATURE SELECTION (RandomForest)

from sklearn.ensemble import RandomForestRegressor

# use one target for feature importance (you can change this)
target = 'profit_margin_calc'

X_train_full = train_data.drop(targets, axis=1)
y_train_full = train_data[target]

# preprocess once to get feature names
preprocessor = build_preprocessor(X_train_full)
X_train_transformed = preprocessor.fit_transform(X_train_full)

# get feature names
cat_cols = X_train_full.select_dtypes(include=['object']).columns
num_cols = X_train_full.select_dtypes(include=[np.number]).columns

ohe = preprocessor.named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(cat_cols)

feature_names = np.concatenate([cat_feature_names, num_cols])

# train RF for importance
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train_transformed, y_train_full)

importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

# select top features (change threshold or top N)
top_features = importances.head(30)['Feature'].tolist()

print("Selected Features:", len(top_features))

# TRANSFORM DATA WITH SELECTED FEATURES

def transform_with_selected_features(X, preprocessor, selected_features):
    X_transformed = preprocessor.transform(X)
    df_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)
    return df_transformed[selected_features]

# RERUN MODELS WITH SELECTED FEATURES

results_selected = []

tscv = TimeSeriesSplit(n_splits=5)

for target in targets:
    print("\nTARGET (SELECTED FEATURES):", target, "\n")

    X_train = train_data.drop(targets, axis=1)
    y_train = train_data[target]

    X_test = test_data.drop(targets, axis=1)
    y_test = test_data[target]

    # transform using selected features
    X_train_sel = transform_with_selected_features(X_train, preprocessor, top_features)
    X_test_sel = transform_with_selected_features(X_test, preprocessor, top_features)

    for name, (model, param_grid) in models_and_params.items():

        grid = GridSearchCV(
            model,
            param_grid,
            cv=tscv,
            scoring='r2',
            n_jobs=-1
        )

        grid.fit(X_train_sel, y_train)

        best_model = grid.best_estimator_

        y_train_pred = best_model.predict(X_train_sel)
        y_test_pred = best_model.predict(X_test_sel)

        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        train_mse = mean_squared_error(y_train, y_train_pred)
        test_mse = mean_squared_error(y_test, y_test_pred)

        results_selected.append({
            'Target': target,
            'Model': name,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Train_MSE': train_mse,
            'Test_MSE': test_mse
        })

        print(name, "Train R2:", round(train_r2, 4), "Test R2:", round(test_r2, 4))

# results
results_selected_df = pd.DataFrame(results_selected)

results_selected_df = results_selected_df.sort_values(
    by=['Target', 'Test_R2'],
    ascending=[True, False]
).reset_index(drop=True)

results_selected_df['Overfit_Gap'] = (
    results_selected_df['Train_R2'] - results_selected_df['Test_R2']
).round(4)

print("\nRESULTS WITH SELECTED FEATURES\n")
print(results_selected_df)

Selected Features: 30

TARGET (SELECTED FEATURES): profit_margin_calc 

Linear Train R2: 0.3193 Test R2: 0.277


ValueError: Invalid parameter 'model' for estimator Ridge(). Valid parameters are: ['alpha', 'copy_X', 'fit_intercept', 'max_iter', 'positive', 'random_state', 'solver', 'tol'].

In [ ]:
# export full results
results_df.to_csv("model_results_full.csv", index=False)

# export best models
best_models.to_csv("best_models.csv", index=False)

# export feature-selected results
results_selected_df.to_csv("model_results_selected_features.csv", index=False)

# export feature importance
importances.to_csv("feature_importance.csv", index=False)

print("\nFiles exported successfully")

In [ ]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

for i, row in best_models.iterrows():
    target = row['Target']
    model_name = row['Model']
    
    # you must store models in loop to access them here
    # assume you saved them in a dictionary called best_model_objects
    
    model = best_model_objects[(target, model_name)]
    
    filename = f"saved_models/best_{target}_{model_name}.pkl"
    joblib.dump(model, filename)

print("All best models saved")